# Qwen3 multilingual learned transformer layer merger

Self-contained Kaggle experiment. Select T4 ×2 and enable Internet. Training and benchmark results are pending until executed. Source modules and tests are embedded below. The supplied GPT-2 architecture is preserved; see the extracted ARCHITECTURE_AUDIT.md.

## 1. Environment setup
Run in a fresh kernel. This cell extracts the complete implementation and installs pinned dependencies without replacing PyTorch.

In [ ]:
import base64, hashlib, io, os, subprocess, sys, zipfile
from pathlib import Path
SOURCE = Path('/kaggle/working/qwen3_merger_source')
SOURCE.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '49149b1a64d3c37c1f4ed5b6366f4d5a90dc4be2c2963c0db232d7c1853b6e90'
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    for name in bundle.namelist():
        target = (SOURCE / name).resolve()
        assert target.is_relative_to(SOURCE.resolve()), name
    bundle.extractall(SOURCE)
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 2. Configuration
Keep the output directory unchanged when resuming. Change token budgets here before a new experiment. Set evaluation_mode="full" later to evaluate the same trained models on full eligible test sets.

In [ ]:
from config import Config
cfg = Config()
cfg.output_dir = '/kaggle/working/Qwen3Merger'
cfg.evaluation_mode = 'pilot'
cfg.save('config.json')
OUT = Path(cfg.output_dir)
OUT.mkdir(parents=True, exist_ok=True)
def stage(name, *extra):
    subprocess.run([sys.executable, 'run.py', name, '--config', 'config.json', *extra], check=True)
print(cfg)


## 3. GPU detection
Normal mode requires two CUDA GPUs. A single-GPU debug run must be explicitly configured.

In [ ]:
from src.trainer import hardware
hardware(cfg)


## 4. Qwen3 loading
Resolve the model and datasets to immutable revisions. The runtime reads the installed Qwen3 decoder signature and validates all parameter shapes.

In [ ]:
from src.trainer import prepare_run
lock = prepare_run(cfg)
print(lock)


## 5. NF4 quantization
The training stage loads actual NF4 tensors before constructing a dense frozen source copy. It separately loads the original FP16 teacher. Quantization presence, hashes and measured storage reduction are mandatory preflight checks.

In [ ]:
from src.quantization import nf4_config
print(nf4_config())


## 6. Weight extraction and patching
128 × 128 matrix patches and 128-element vectors; 978 tokens per source, 1,956 per pair for the expected configuration. All 11 decoder parameter families are covered.

In [ ]:
import json
print(json.dumps(json.loads(Path("reference/architecture_counts.json").read_text()), indent=2))


## 7. Merger architecture
Six bidirectional attention blocks, shared shape encoders/decoders, matched-patch fusion, and second-block anchoring. Initial corrections are exactly zero. The merger has 23,676,288 parameters versus 39,391,168 in the reference; this capacity change is documented.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path("ARCHITECTURE_AUDIT.md").read_text()))


## 8. Wiki-40B preparation
Discover every available language and prepare balanced token budgets from disjoint TRAIN article partitions. Also freeze benchmark prompts and demonstration IDs before model comparisons.

In [ ]:
stage('prepare')


## 9. Training setup and preflight tests
Run the source-level tests and actual benchmark smoke tests first. The training stage then performs all model/NF4/gradient/dual-GPU/export checks before its first optimizer update; a critical failure stops the run.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)
stage('benchmark-preflight')


## 10. Dual-GPU training
Both GPU replicas contribute gradients. The five-merge batch tuner measures memory with Adam state allocated. Resume uses latest.pt automatically. Save the output directory between Kaggle sessions.

In [ ]:
stage('train')


## 11. Validation and checkpoint selection
Validation occurs inside training and uses equal compression-level CE gaps against matching drop-first. Benchmark test answers never select checkpoints.

In [ ]:
import pandas as pd
display(pd.read_csv(OUT / 'logs/validation.csv'))
print((OUT / 'manifests/training_complete.json').read_text())


## 12. Layer-pair ranking
Rank all 27 original adjacent pairs on the independent holdout. Select raw and guarded compatible plans separately, then measure each whole plan. This stage also exports and reload-checks models; it can run in a later session.

In [ ]:
stage('rank-export')
display(pd.read_csv(OUT / 'rankings/single_pairs.csv'))


## 13. Model compression and export
Exports contain ordinary Qwen3 models and the original tokenizer. Control definitions are saved separately and reconstructed during evaluation to save disk space. Evaluation creates separately labelled serialized NF4 deployment models.

In [ ]:
exports = json.loads((OUT / 'manifests/exports.json').read_text())
display(pd.DataFrame(exports)[['name', 'layers', 'parameters', 'stored_bytes', 'representation']])


## 14. MMMLU evaluation
All variants use the frozen few-shot prompts and selected IDs. Predictions are incrementally saved; rerunning resumes missing examples. Pilot scores are sampled experimental scores, not full official results.

In [ ]:
stage('benchmark', '--benchmark', 'MMMLU')


## 15. INCLUDE evaluation
All variants use the frozen few-shot prompts and selected IDs. Predictions are incrementally saved; rerunning resumes missing examples. Pilot scores are sampled experimental scores, not full official results.

In [ ]:
stage('benchmark', '--benchmark', 'INCLUDE')


## 16. MGSM evaluation
All variants use the frozen few-shot prompts and selected IDs. Predictions are incrementally saved; rerunning resumes missing examples. Pilot scores are sampled experimental scores, not full official results.

In [ ]:
stage('benchmark', '--benchmark', 'MGSM')


## 17. Result comparison
Write per-language and aggregate tables, paired bootstrap intervals, coverage, subgroup scores and measured efficiency. Incomplete comparisons fail rather than silently changing the paired sample.

In [ ]:
stage('report')
RESULTS = OUT / 'benchmark_results' / cfg.evaluation_mode
display(pd.read_csv(RESULTS / 'aggregate.csv'))
display(pd.read_csv(RESULTS / 'paired_differences.csv'))


## 18. Charts
Figures use recorded measurements only. PNG (300 DPI), PDF and underlying CSV files are retained.

In [ ]:
from IPython.display import Image
for path in sorted((OUT / 'charts').glob('*.png')):
    display(Image(filename=str(path)))


## 19. Final report and downloads
Save the entire output folder with Kaggle Save Version. The small archive below contains analysis artifacts; large model exports and checkpoints remain in the output folder for separate download.

In [ ]:
from IPython.display import FileLink
display(Markdown((OUT / 'REPORT.md').read_text()))
analysis_zip = OUT / 'analysis_outputs.zip'
with zipfile.ZipFile(analysis_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['logs', 'manifests', 'rankings', 'benchmark_results', 'charts']:
        for path in (OUT / folder).rglob('*'):
            if path.is_file(): z.write(path, path.relative_to(OUT))
    z.write(OUT / 'REPORT.md', 'REPORT.md')
display(FileLink(str(analysis_zip)))
print('Save checkpoints/ and model_exports/ separately through Kaggle outputs.')
